In [ ]:
import os
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T

from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm
from diffusers import AutoencoderKL

In [ ]:
if torch.cuda.is_available():
    device = torch.device('cuda')
#elif torch.backends.mps.is_available():
#    device = torch.device('mps')
else:
    device = torch.device('cpu')

In [ ]:
IMG_SIZE = 64
BATCH_SIZE_IMG = 32

tfm = T.Compose([
    T.Resize(IMG_SIZE, interpolation=T.InterpolationMode.BICUBIC),
    T.CenterCrop(IMG_SIZE),
    T.ToTensor(),
])

img_dataset = torchvision.datasets.STL10(
    root="datasets/",
    split="unlabeled",
    download=True,
    transform=tfm,
)

img_loader = DataLoader(
    img_dataset,
    batch_size=BATCH_SIZE_IMG,
    shuffle=False,
    num_workers=4,
    pin_memory=True if device.type == 'cuda' else False,
)

In [ ]:
vae = AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-mse")
vae = vae.to(device).eval()
vae.requires_grad_(False)

scaling = vae.config.scaling_factor
scaling

In [ ]:
LATENT_PATH = "datasets/stl10/latents_stl10_sdvae.pt"
MAX_LATENTS = 20000  # set None for all, but this is enough for playing

if not os.path.exists(LATENT_PATH):
    latents = []
    n_seen = 0

    for x, _ in tqdm(img_loader):
        x = x.to(device)
        x = 2.0 * x - 1.0  # [0,1] -> [-1,1]

        with torch.no_grad():
            posterior = vae.encode(x).latent_dist
            z = posterior.sample() * scaling

        latents.append(z.cpu().half())
        n_seen += x.shape[0]

        if MAX_LATENTS is not None and n_seen >= MAX_LATENTS:
            break

    latents = torch.cat(latents, dim=0)
    if MAX_LATENTS is not None:
        latents = latents[:MAX_LATENTS]

    torch.save(latents, LATENT_PATH)

else:
    latents = torch.load(LATENT_PATH)

latents = latents.float()
latents.shape

In [ ]:
BATCH_SIZE_LATENT = 64

latent_dataset = TensorDataset(latents)

latent_loader = DataLoader(
    latent_dataset,
    batch_size=BATCH_SIZE_LATENT,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True if device.type == 'cuda' else False,
)

In [ ]:
z_mean = latents.mean(dim=(0, 2, 3), keepdim=True)
z_std = latents.std(dim=(0, 2, 3), keepdim=True).clamp_min(1e-6)

def normalize_z(z):
    return (z - z_mean.to(z.device)) / z_std.to(z.device)

def unnormalize_z(z):
    return z * z_std.to(z.device) + z_mean.to(z.device)